# Qorvix — CUDA backend test on a real GPU

Builds Qorvix with the real CUDA backend (`nvcc`) and runs its GPU self-tests on the actual
device. Use this to verify CUDA **execution** — the dev box and CI only compile-check it.

## ⚠️ Use a GPU runtime, NOT a TPU

CUDA runs only on NVIDIA GPUs. Set **Runtime → Change runtime type → Hardware accelerator = T4
GPU** (or any GPU). A **TPU** runtime has no `nvcc` / no CUDA device and the first cell will stop
with an error.

Run the cells in order. Each step is separate so any failure is easy to spot.

## 1. Check for an NVIDIA GPU

In [25]:
import subprocess, sys
ok = subprocess.run(['bash','-lc','nvidia-smi >/dev/null 2>&1']).returncode == 0
if not ok:
    print('ERROR: no NVIDIA GPU detected.')
    print('Colab: Runtime > Change runtime type > Hardware accelerator = T4 GPU (NOT TPU), then rerun.')
    raise SystemExit(1)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
!nvcc --version | grep -i release || echo 'WARNING: nvcc not found (GPU but no CUDA toolkit)'

Tesla T4, 580.82.07, 15360 MiB
Cuda compilation tools, release 12.8, V12.8.93


## 2. Install build tools (ninja, gcc-12, recent CMake)

In [26]:
!apt-get -qq update >/dev/null 2>&1
!apt-get -qq install -y ninja-build g++-12 gcc-12 >/dev/null 2>&1
!pip -q install --upgrade cmake >/dev/null 2>&1
!cmake --version | head -1

cmake version 4.4.0


## 3. Clone the repository

In [27]:
!rm -rf /content/qorvix
!git clone --depth 1 https://github.com/Boominathan2355/QorVix.git /content/qorvix
!git -C /content/qorvix log --oneline -1

Cloning into '/content/qorvix'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 171 (delta 0), reused 136 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 160.09 KiB | 1.13 MiB/s, done.
ce934e7 (grafted, HEAD -> main, origin/main, origin/HEAD) Add gpu-check cell to the Colab notebook (real TinyLlama on GPU)


## 4. Configure with CUDA enabled

No vcpkg needed: the CUDA executable and its libraries use only `nvcc` + a C++23 host compiler
(Boost/Catch2 are just for the HTTP API and unit tests, which are turned off here).

In [28]:
!cd /content/qorvix && CC=gcc-12 CXX=g++-12 cmake -S . -B build-cuda -G Ninja \
  -DCMAKE_BUILD_TYPE=Release \
  -DQORVIX_ENABLE_CUDA=ON \
  -DQORVIX_BUILD_TESTS=OFF \
  -DCMAKE_CUDA_HOST_COMPILER=g++-12

-- The CXX compiler identification is GNU 12.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/g++-12 - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Looking for a CUDA compiler
-- Looking for a CUDA compiler - /usr/local/cuda/bin/nvcc
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- qorvix_cuda: building the real CUDA backend (nvcc).
-- qorvix_runtime: OpenMP enabled
-- Configuring done (8.5s)
-- Generat

## 5. Build

Watch for `qorvix_cuda: building the real CUDA backend (nvcc).` in the configure output above, and
for `Building CUDA object ... cuda_backend.cu.o` here.

In [29]:
!cmake --build /content/qorvix/build-cuda -j$(nproc)

[18/41] Building CUDA object cuda/CMakeFiles/qorvix_cuda.dir/src/cuda_backend.cu.oK.oK
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
/content/qorvix/cuda/src/cuda_backend.cu(1188): warning #177-D: variable "qDim" was declared but never referenced
      const int qDim = cfg_.nHeads * cfg_.headDim, hd = cfg_.headDim, mx = cfg_.maxSeq;
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

In static member function ‘static constexpr _Tp* std::__copy_move<_IsMove, true, std::random_access_iterator_tag>::__copy_m(const _Tp*, const _Tp*, _Tp*) [with _Tp = float; bool _IsMove = false]’,
    inlined from ‘constexpr _OI std::__copy_move_a2(_II, _II, _OI) [with bool _IsMove = false; _II = float*; _OI = float*]’ at /usr/include/c++/11/bits/stl_algobase.h:494:141,
    inlined from ‘constexpr _OI std::__copy_move_a1(_

## 6. Run the GPU self-tests on the real device

`qorvix gpu` enumerates the device and runs a custom scale kernel (host→device→kernel→host) and a
cuBLAS SGEMM, checking results on the host. Exit code is nonzero if any self-test fails.

In [30]:
!/content/qorvix/build-cuda/core/qorvix gpu

CUDA support: built in.
Devices: 1

  [0] Tesla T4
      compute capability : 7.5
      SMs                : 40
      memory (free/total): 14 GB / 15 GB

Self-test (scale kernel): PASS - scale kernel host<->device round-trip verified over 1024 floats
Self-test (cuBLAS GEMM):  PASS - cuBLAS SGEMM (A=I) verified on a 4x4 matrix
Self-test (Q8_0 matmul):  PASS - GPU Q8_0 matmul matches host reference (max err 0.000000); 4096x4096 Q8_0 GEMV: 107 GFLOP/s, 57 GB/s
Self-test (Q4_K matmul):  PASS - GPU Q4_K matmul matches host reference (rel err 0.000000)
Self-test (Q6_K matmul):  PASS - GPU Q6_K matmul matches host reference (rel err 0.000000)
Self-test (forward ops):  PASS - rmsnorm/rope/swiglu/add match CPU (max err 0.000000)
Self-test (attention):    PASS - GQA decode attention matches CPU (max err 0.000000)
Self-test (GPU forward):  PASS - GPU forward pass (2-layer, 3 positions) matches CPU reference (max err 0.000000)


## 7. Real-model GPU inference: GPU vs CPU logits

Downloads **TinyLlama 1.1B Q4_K_M** (~670 MB) and runs `qorvix gpu-check`, which executes the
forward pass on both the CPU runtime and the GPU (weights + KV cache resident in VRAM) and
compares the logits. PASS = real GPU inference matches the CPU reference on real weights.

In [35]:
!mkdir -p /content/qorvix/models
!curl -fsSL -o /content/qorvix/models/tinyllama.gguf https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
!/content/qorvix/build-cuda/core/qorvix gpu-check /content/qorvix/models/tinyllama.gguf

Uploading weights to VRAM and building GPU model...
Comparing GPU vs CPU logits over 6 prompt tokens...

GPU vs CPU logits:  max abs err 5.76973e-05, rel err 3.71482e-06
Argmax agrees at every position: yes
RESULT: PASS - GPU forward matches the CPU runtime.


## 8. Generate on the GPU (tokens/sec benchmark)

Runs real text generation with the transformer forward pass on the GPU (`generate --gpu`) and
reports tokens/sec. Compare against the CPU runtime's ~0.7 tok/s.

In [36]:
!/content/qorvix/build-cuda/core/qorvix generate /content/qorvix/models/tinyllama.gguf \n
  --gpu --prompt 'The capital of France is' --temp 0 --max 40

IndentationError: unexpected indent (112900555.py, line 2)

## What this proves

The `qorvix gpu` self-tests confirm every GPU kernel (Q4_K/Q6_K/Q8_0 matmul, RMSNorm, RoPE,
attention, SwiGLU) and the assembled forward pass execute correctly on this device.

`qorvix gpu-check` goes further: it runs a **real GGUF model** (TinyLlama) through the on-GPU
forward pass and confirms the logits match the CPU runtime. A PASS there means real GPU
inference is correct. The remaining step is `generate --gpu` + a tokens/sec benchmark.

Copy the output back to the chat for interpretation.

In [33]:
!curl -fsSL https://raw.githubusercontent.com/Boominathan2355/QorVix/main/scripts/colab_cuda_test.sh | bash
# then, in the same runtime:
!mkdir -p /content/qorvix/models
!curl -fsSL -o /content/qorvix/models/tinyllama.gguf https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
!/content/qorvix/build-cuda/core/qorvix gpu-check /content/qorvix/models/tinyllama.gguf


==================== Qorvix CUDA GPU test ====================
Tesla T4, 580.82.07, 15360 MiB
Cuda compilation tools, release 12.8, V12.8.93
---- installing build tools (ninja, gcc-12) ----
cmake: cmake version 4.4.0
---- cloning Qorvix into /content/qorvix ----
Cloning into '/content/qorvix'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 171 (delta 0), reused 136 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 160.09 KiB | 1.20 MiB/s, done.
---- configuring (QORVIX_ENABLE_CUDA=ON) ----
-- The CXX compiler identification is GNU 12.3.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/g++-12 - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Looking for a CUDA compiler
-- Looking for a CUDA compiler - /usr/local/cuda/bin/nvcc
-- The CUDA compiler identi